## Without web search

In [1]:
%run init_env.py

Added to Python path: C:\git\cs5305
Environment initialization completed successfully.


In [2]:
from azure_openai_llm import create_azure_llm
from langchain.agents import create_agent
from langchain.messages import HumanMessage  

llm = create_azure_llm()

agent = create_agent(
    model=llm
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [3]:
agent.invoke(
    {"messages": [HumanMessage(content="who is current mayor of lahore?")]}
)['messages'][-1].content

'As of June 2024, the current mayor of Lahore is **Mian Yousuf Salahuddin**.'

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)
response['messages'][-1].content

'My training includes information up until November 2023. If you have questions about recent events or developments after that date, I might not have the latest details. How can I assist you today?'

## Add web search tool

In [5]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of Lahore?")

{'query': 'Who is the current mayor of Lahore?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Lahore',
   'title': 'Lahore - Wikipedia',
   'content': 'Mayor of Lahore is the elected head of the Metropolitan Corporation of Lahore. The mayor is directly elected in municipal elections every four years',
   'score': 0.9995345,
   'raw_content': None},
  {'url': 'https://en.wikipedia.org/wiki/Mayor_of_Lahore',
   'title': 'Mayor of Lahore - Wikipedia',
   'content': 'Mayors of Lahore ; Mian Amir Mehmood, 2005, 2009 ; Administrator system implemented 2010–2016 ; Mubashar Javed, 2016, 2021 ; Administrator system implemented',
   'score': 0.99933845,
   'raw_content': None},
  {'url': 'https://www.facebook.com/Lord.Mayor.Lahore/',
   'title': 'Lord Mayor Lahore - Facebook',
   'content': 'Samiullah Chaudhary is with M Qasim Khalifah in Lahore, Pakistan. Feb 19, 2016\U000f078b\U000f17e0. Today meeting with my Reverent Quaid C

In [6]:
agent = create_agent(
    llm,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of Lahore?")

response = agent.invoke(
    {"messages": [question]}
)

In [7]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of Lahore?', additional_kwargs={}, response_metadata={}, id='2b54d10d-3394-4dd3-803d-98f5891f027d'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 51, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DSSbJ8OZCWLpiw3s7ovlyF5Pfr2So', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'saf

In [8]:
response['messages'][-1].content

'The most recent information indicates that Mubashir Javed of the Pakistan Muslim League (N) was elected mayor of Lahore in 2016. However, the search results do not provide confirmation if he is still the current mayor. Would you like me to look for the latest updates on the current mayor of Lahore?'

In [9]:
pprint(response["messages"][1].tool_calls)

[{'args': {'query': 'current mayor of Lahore'},
  'id': 'call_wR0vPLXhrq6huqYGo9oGzw4u',
  'name': 'web_search',
  'type': 'tool_call'}]


In [10]:
from langchain_core.tracers.context import tracing_v2_enabled

with tracing_v2_enabled() as ctx:
    question = HumanMessage(content="How is the weather today in Lahore?")

    response = agent.invoke(
        {"messages": [question]}
    )
    pprint(response['messages'][-1].content)
    url = ctx.get_run_url()
    print(f"View the trace at: {url}")

('The current weather in Lahore is overcast with a temperature of around '
 '18.4°C (65.1°F). The humidity is at 77%, and there is a light wind blowing '
 'from the southeast at about 4.7 km/h (2.9 mph). There is no precipitation '
 'reported at the moment. If you want the forecast for later today, there is a '
 'chance of thunderstorms with temperatures ranging approximately from 59°F to '
 '74°F.')
View the trace at: https://eu.smith.langchain.com/o/ed681c67-0bab-40c3-ad5a-fe23a6c3ddb4/projects/p/c01085aa-1fe5-450f-995e-347c849660ab/r/019d6e81-0231-75e0-a4c3-7521b7f8ce03?poll=true
